# Tối ưu hoá đọc dữ liệu trong Spark — Notebook thực hành

Notebook này giúp bạn **thực sự cảm nhận** sự khác biệt giữa các kỹ thuật tối ưu đọc dữ liệu thông qua đo lường cụ thể (thời gian, lượng I/O, kế hoạch thực thi).

## Nội dung

1. **Setup**: tạo SparkSession và sinh dữ liệu mẫu (~1 triệu dòng, 20 cột)
2. **CSV vs Parquet**: so sánh kích thước file và tốc độ đọc
3. **Predicate Pushdown**: đo lường filter trên Parquet vs CSV
4. **Column Pruning**: so sánh `select` 2 cột vs đọc toàn bộ
5. **Partition Pruning**: ghi với `partitionBy` và đo I/O
6. **Kết hợp cả 3**: query thực tế trên data lake mô phỏng
7. **Internal vs External Table**: thực nghiệm `DROP TABLE`

## Yêu cầu

```bash
pip install pyspark==3.5.0
```

Notebook chạy được trên máy local — không cần cluster.

---
## 1. Setup — SparkSession và dữ liệu mẫu

In [1]:
import os
import time
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Khởi tạo SparkSession
spark = (SparkSession.builder
         .appName("ReadOptimization")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "4")  # giảm cho local
         .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

ModuleNotFoundError: No module named 'pyspark'

Working dir: data/demo-module-4.4


In [ ]:
# Sinh dữ liệu mẫu: 100 dòng, 20 cột, mô phỏng bảng sales
# - year (2023, 2024, 2025), month (1-12)
# - city (5 thành phố), category (10 loại)
# - 16 cột số phụ để mô phỏng bảng nhieu cot — testing column pruning

N_ROWS = 100

df = (spark.range(0, N_ROWS)
      .withColumn("year",     (F.col("id") % 3 + 2023).cast("int"))
      .withColumn("month",    (F.col("id") % 12 + 1).cast("int"))
      .withColumn("city",     F.element_at(F.array(
          F.lit("HCM"), F.lit("HN"), F.lit("DN"), F.lit("CT"), F.lit("HP")
      ), (F.col("id") % 5 + 1).cast("int")))
      .withColumn("category", F.concat(F.lit("cat_"), (F.col("id") % 10).cast("string")))
      .withColumn("amount",   (F.rand() * 1000).cast("double"))
      .withColumn("quantity", (F.col("id") % 100).cast("int")))

# Thêm 14 cột số phụ để bảng nhieu cot hơn — quan trọng cho demo column pruning
for i in range(14):
    df = df.withColumn(f"extra_col_{i}", (F.rand() * 10000).cast("double"))

print(f"Tổng số cột: {len(df.columns)}")
df.printSchema()
df.show(3, truncate=False)

Tổng số cột: 21
root
 |-- id: long (nullable = false)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- extra_col_0: double (nullable = false)
 |-- extra_col_1: double (nullable = false)
 |-- extra_col_2: double (nullable = false)
 |-- extra_col_3: double (nullable = false)
 |-- extra_col_4: double (nullable = false)
 |-- extra_col_5: double (nullable = false)
 |-- extra_col_6: double (nullable = false)
 |-- extra_col_7: double (nullable = false)
 |-- extra_col_8: double (nullable = false)
 |-- extra_col_9: double (nullable = false)
 |-- extra_col_10: double (nullable = false)
 |-- extra_col_11: double (nullable = false)
 |-- extra_col_12: double (nullable = false)
 |-- extra_col_13: double (nullable = false)



+---+----+-----+----+--------+------------------+--------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+-----------------+
|id |year|month|city|category|amount            |quantity|extra_col_0       |extra_col_1       |extra_col_2       |extra_col_3       |extra_col_4      |extra_col_5       |extra_col_6       |extra_col_7       |extra_col_8       |extra_col_9       |extra_col_10      |extra_col_11      |extra_col_12     |extra_col_13     |
+---+----+-----+----+--------+------------------+--------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+-----------------+
|0  |2023|1    |HCM |cat_0   |255.

---
## 2. CSV vs Parquet — Kích thước file và tốc độ đọc

**Mục tiêu**: thấy được Parquet vừa **nhẹ hơn** (nhờ nén columnar) vừa **đọc nhanh hơn** (nhờ metadata + columnar layout).

In [ ]:
# Helper: đo thời gian thực thi một action
def timed(name, func):
    t0 = time.time()
    result = func()
    dt = time.time() - t0
    print(f"  [{name}] {dt:.3f}s  →  result = {result}")
    return dt

# Helper: tính dung lượng thư mục
def dir_size_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            if os.path.exists(fp):
                total += os.path.getsize(fp)
    return total / (1024 * 1024)

In [ ]:
# Ghi data ra cả 2 format để so sánh
BASE = "/opt/workspace/data/demo-module-4.4"
csv_path     = f"file://{BASE}/sales_csv"
parquet_path = f"file://{BASE}/sales_parquet"

print("Ghi CSV ...")
df.write.mode("overwrite").option("header", "true").csv(csv_path)

print("Ghi Parquet ...")
df.write.mode("overwrite").parquet(parquet_path)


Ghi CSV ...


Ghi Parquet ...



Dung lượng CSV    :     0.00 MB
Dung lượng Parquet:     0.00 MB


ZeroDivisionError: float division by zero

In [ ]:
# Đo thời gian count toàn bảng
print("COUNT toàn bảng:")
t_csv = timed("CSV    ", lambda: spark.read.option("header", "true").csv(csv_path).count())
t_pq  = timed("Parquet", lambda: spark.read.parquet(parquet_path).count())
print(f"\n→ Parquet nhanh hơn CSV {t_csv/t_pq:.1f}x lần")

**Quan sát**:
- Parquet nén tốt hơn nhiều (~5-10x nhỏ hơn CSV) nhờ tổ chức theo cột + dictionary encoding.
- Khi count, Parquet chỉ cần đọc metadata footer → cực nhanh; CSV phải scan toàn bộ file.

---
## 3. Predicate Pushdown — Filter được đẩy xuống tầng file

**Mục tiêu**: thấy `PushedFilters` trong physical plan và đo tốc độ filter.

In [23]:
# Xem physical plan để kiểm tra PushedFilters
df_pq = spark.read.parquet(parquet_path)
filtered_pq = df_pq.filter((F.col("year") == 2024) & (F.col("city") == "HCM"))

print("="*70)
print("PHYSICAL PLAN (Parquet) — chú ý dòng PushedFilters:")
print("="*70)
filtered_pq.explain(mode="formatted")


[Stage 34:>                                                         (0 + 1) / 1]



PHYSICAL PLAN (Parquet) — chú ý dòng PushedFilters:
== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [21]: [id#1079L, year#1080, month#1081, city#1082, category#1083, amount#1084, quantity#1085, extra_col_0#1086, extra_col_1#1087, extra_col_2#1088, extra_col_3#1089, extra_col_4#1090, extra_col_5#1091, extra_col_6#1092, extra_col_7#1093, extra_col_8#1094, extra_col_9#1095, extra_col_10#1096, extra_col_11#1097, extra_col_12#1098, extra_col_13#1099]
Batched: true
Location: InMemoryFileIndex [file:/opt/workspace/data/demo-module-4.4/sales_parquet]
PushedFilters: [IsNotNull(year), IsNotNull(city), EqualTo(year,2024), EqualTo(city,HCM)]
ReadSchema: struct<id:bigint,year:int,month:int,city:string,category:string,amount:double,quantity:int,extra_col_0:double,extra_col_1:double,extra_col_2:double,extra_col_3:double,extra_col_4:double,extra_col_5:double,extra_col_6:double,extra_col_7:double,extra_col_8:double,extra_col_9:double,extra_col_

In [21]:
# So sánh với CSV — CSV KHÔNG hỗ trợ pushdown
df_csv = spark.read.option("header", "true").csv(csv_path)
filtered_csv = df_csv.filter((F.col("year") == 2024) & (F.col("city") == "HCM"))

print("="*70)
print("PHYSICAL PLAN (CSV) — KHÔNG có PushedFilters thực sự:")
print("="*70)
filtered_csv.explain(mode="formatted")

PHYSICAL PLAN (CSV) — KHÔNG có PushedFilters thực sự:
== Physical Plan ==
*(1) Filter (((isnotnull(year#719) AND isnotnull(city#721)) AND (cast(year#719 as int) = 2024)) AND (city#721 = HCM))
+- FileScan csv [id#718,year#719,month#720,city#721,category#722,amount#723,quantity#724,extra_col_0#725,extra_col_1#726,extra_col_2#727,extra_col_3#728,extra_col_4#729,extra_col_5#730,extra_col_6#731,extra_col_7#732,extra_col_8#733,extra_col_9#734,extra_col_10#735,extra_col_11#736,extra_col_12#737,extra_col_13#738] Batched: false, DataFilters: [isnotnull(year#719), isnotnull(city#721), (cast(year#719 as int) = 2024), (city#721 = HCM)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/opt/workspace/data/demo-module-4.4/sales_csv], PartitionFilters: [], PushedFilters: [IsNotNull(year), IsNotNull(city), EqualTo(city,HCM)], ReadSchema: struct<id:string,year:string,month:string,city:string,category:string,amount:string,quantity:stri...




'file:///opt/workspace/data/demo-module-4.4/sal'

In [ ]:
# Đo thời gian filter
print("FILTER year=2024 AND city='HCM':")
t_csv_f = timed("CSV    ", lambda: df_csv.filter((F.col("year") == 2024) & (F.col("city") == "HCM")).count())
t_pq_f  = timed("Parquet", lambda: df_pq.filter((F.col("year") == 2024) & (F.col("city") == "HCM")).count())
print(f"\n→ Predicate pushdown giúp Parquet nhanh hơn {t_csv_f/t_pq_f:.1f}x lần")

**Quan sát**: trong physical plan của Parquet, bạn sẽ thấy:
```
PushedFilters: [IsNotNull(year), IsNotNull(city), EqualTo(year,2024), EqualTo(city,HCM)]
```
Đây là **bằng chứng** Spark đã đẩy filter xuống tận tầng file. Với CSV, filter chỉ được áp dụng sau khi đọc xong toàn bộ.

---
## 4. Column Pruning — Chỉ đọc cột cần

**Mục tiêu**: bảng có 20 cột, nhưng query chỉ cần 2 → đo lượng giảm I/O.

In [22]:
# Test với Parquet — column pruning hoạt động
print("PARQUET — đọc 20 cột vs 2 cột:")
t_all_pq = timed("All 20 cols", lambda: spark.read.parquet(parquet_path).select("*").count())
t_2_pq   = timed("2 cols     ", lambda: spark.read.parquet(parquet_path).select("city", "amount").count())

print("\nCSV — column pruning KHÔNG hoạt động (vẫn phải scan cả dòng):")
t_all_csv = timed("All 20 cols", lambda: spark.read.option("header", "true").csv(csv_path).select("*").count())
t_2_csv   = timed("2 cols     ", lambda: spark.read.option("header", "true").csv(csv_path).select("city", "amount").count())

print(f"\n→ Parquet: chọn 2 cột nhanh hơn chọn 20 cột {t_all_pq/t_2_pq:.1f}x lần")
print(f"→ CSV    : chọn 2 cột vs 20 cột tương đương ({t_all_csv/t_2_csv:.1f}x) — không có pruning")

PARQUET — đọc 20 cột vs 2 cột:


  [All 20 cols] 5.380s  →  result = 100


  [2 cols     ] 2.839s  →  result = 100

CSV — column pruning KHÔNG hoạt động (vẫn phải scan cả dòng):
  [All 20 cols] 1.579s  →  result = 100


  [2 cols     ] 6.742s  →  result = 100

→ Parquet: chọn 2 cột nhanh hơn chọn 20 cột 1.9x lần
→ CSV    : chọn 2 cột vs 20 cột tương đương (0.2x) — không có pruning


In [24]:
# Xem physical plan để confirm — chú ý phần "ReadSchema"
print("="*70)
print("ReadSchema chỉ chứa 2 cột được select:")
print("="*70)
spark.read.parquet(parquet_path).select("city", "amount").explain(mode="formatted")

ReadSchema chỉ chứa 2 cột được select:



[Stage 35:>                                                         (0 + 1) / 1]



== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [city#1125,amount#1127] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/opt/workspace/data/demo-module-4.4/sales_parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<city:string,amount:double>




In [25]:
spark.read.option("header", "true").csv(csv_path).select("city", "amount").explain(mode="formatted")

== Physical Plan ==
Scan csv  (1)


(1) Scan csv 
Output [2]: [city#1187, amount#1189]
Batched: false
Location: InMemoryFileIndex [file:/opt/workspace/data/demo-module-4.4/sales_csv]
ReadSchema: struct<city:string,amount:string>




== Optimized Logical Plan ==
Project [city#1505, amount#1507], Statistics(sizeInBytes=13.2 KiB)
+- Relation [id#1502L,year#1503,month#1504,city#1505,category#1506,amount#1507,quantity#1508,extra_col_0#1509,extra_col_1#1510,extra_col_2#1511,extra_col_3#1512,extra_col_4#1513,extra_col_5#1514,extra_col_6#1515,extra_col_7#1516,extra_col_8#1517,extra_col_9#1518,extra_col_10#1519,extra_col_11#1520,extra_col_12#1521,extra_col_13#1522] parquet, Statistics(sizeInBytes=68.8 KiB)

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [city#1505,amount#1507] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/opt/workspace/data/demo-module-4.4/sales_parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<city:string,amount:double>





[Stage 49:>                                                         (0 + 1) / 1]



== Optimized Logical Plan ==
Project [city#1567, amount#1569], Statistics(sizeInBytes=3.5 KiB)
+- Relation [id#1564,year#1565,month#1566,city#1567,category#1568,amount#1569,quantity#1570,extra_col_0#1571,extra_col_1#1572,extra_col_2#1573,extra_col_3#1574,extra_col_4#1575,extra_col_5#1576,extra_col_6#1577,extra_col_7#1578,extra_col_8#1579,extra_col_9#1580,extra_col_10#1581,extra_col_11#1582,extra_col_12#1583,extra_col_13#1584] csv, Statistics(sizeInBytes=30.8 KiB)

== Physical Plan ==
FileScan csv [city#1567,amount#1569] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/opt/workspace/data/demo-module-4.4/sales_csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<city:string,amount:string>





---
## 5. Partition Pruning — Bỏ qua thư mục không cần

**Mục tiêu**: ghi data với `partitionBy("year", "month")` và đo lường khi filter trên cột partition.

In [ ]:
# Ghi data có partition
partitioned_local = f"{BASE}/sales_partitioned"
partitioned_path = f"file://{partitioned_local}"

print("Ghi data với partitionBy(year, month) ...")
df.write.mode("overwrite").partitionBy("year", "month").parquet(partitioned_path)

# Xem cấu trúc thư mục được tạo:
print("\nCấu trúc thư mục được tạo:")
for root, dirs, files in os.walk(partitioned_local):
    level = root.replace(partitioned_local, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 2:  # không in file để gọn
        break

In [ ]:
# So sánh: filter trên partitioned vs non-partitioned
print("FILTER year=2024 AND month=6:")

# Không partition — phải scan toàn bộ file
t_no_part = timed("Không partition", 
    lambda: spark.read.parquet(parquet_path)
             .filter((F.col("year") == 2024) & (F.col("month") == 6)).count())

# Có partition — chỉ đọc 1 thư mục
t_part = timed("Có partition   ",
    lambda: spark.read.parquet(partitioned_path)
             .filter((F.col("year") == 2024) & (F.col("month") == 6)).count())

print(f"\n→ Partition pruning nhanh hơn {t_no_part/t_part:.1f}x lần")
print(f"→ Số partition được đọc: 1/36 = {1/36*100:.1f}% dữ liệu")

In [ ]:
# Xem physical plan — chú ý PartitionFilters
print("="*70)
print("PartitionFilters tách biệt với PushedFilters:")
print("="*70)
(spark.read.parquet(partitioned_path)
  .filter((F.col("year") == 2024) & (F.col("month") == 6) & (F.col("city") == "HCM"))
  .explain(False))

**Quan sát**: physical plan chia thành 2 nhóm filter:
- `PartitionFilters: [year=2024, month=6]` → Spark chỉ vào đúng thư mục `year=2024/month=6/`.
- `PushedFilters: [EqualTo(city,HCM)]` → trong các file của thư mục đó, Spark còn pushdown thêm filter trên `city` xuống Parquet.

### ⚠️ Demo "small file problem"

Hãy thử partition trên cột high-cardinality để **thấy** vấn đề:

In [ ]:
# DEMO: partition theo cột có cardinality cao = THẢM HOẠ
# Ở đây ta partition theo `quantity` (100 giá trị) — đã là quá nhiều với 1M rows

bad_partition_local = f"{BASE}/sales_bad_partition"
bad_partition_path = f"file://{bad_partition_local}"
print("Ghi với partitionBy(quantity) — 100 giá trị khác nhau...")
df.write.mode("overwrite").partitionBy("quantity").parquet(bad_partition_path)

# Đếm số file
n_files = sum(len(files) for _, _, files in os.walk(bad_partition_local))
size_mb = dir_size_mb(bad_partition_local)
print(f"\nSố file tạo ra : {n_files}")
print(f"Dung lượng     : {size_mb:.2f} MB")
print(f"Trung bình/file: {size_mb*1024/n_files:.1f} KB  ← QUÁ NHỎ!")

# So với partition tốt
n_files_good = sum(len(files) for _, _, files in os.walk(partitioned_local))
size_mb_good = dir_size_mb(partitioned_local)
print(f"\nĐể so sánh — partition theo (year, month):")
print(f"Số file        : {n_files_good}")
print(f"Trung bình/file: {size_mb_good*1024/n_files_good:.1f} KB")

**Bài học**: partition sai → hàng trăm/nghìn file nhỏ → đọc chậm, overhead lớn. Quy tắc: **mỗi partition nên có ít nhất 100MB-1GB data**.

---
## 6. Kết hợp cả 3 kỹ thuật

**Mục tiêu**: chạy 1 query thực tế kết hợp partition pruning + column pruning + predicate pushdown.

In [ ]:
# Query thực tế: doanh thu thành phố HCM trong tháng 6/2024
print("Query: tổng doanh thu HCM, tháng 6/2024")
print("="*70)

# Cách CHẬM: đọc CSV, không có gì tối ưu
def query_slow():
    return (spark.read.option("header", "true").csv(csv_path)
            .filter((F.col("year") == 2024) & (F.col("month") == 6) & (F.col("city") == "HCM"))
            .agg(F.sum(F.col("amount").cast("double")).alias("total"))
            .collect()[0]["total"])

# Cách NHANH: Parquet + partition + column pruning
def query_fast():
    return (spark.read.parquet(partitioned_path)
            .filter((F.col("year") == 2024) & (F.col("month") == 6) & (F.col("city") == "HCM"))
            .select("amount")  # chỉ select cột cần
            .agg(F.sum("amount").alias("total"))
            .collect()[0]["total"])

t_slow = timed("CSV (không tối ưu)         ", query_slow)
t_fast = timed("Parquet + partition + pruning", query_fast)

print(f"\n🚀 Tổng tăng tốc: {t_slow/t_fast:.1f}x lần")

In [ ]:
# Xem plan đầy đủ — cả 3 kỹ thuật cùng hoạt động
print("="*70)
print("PHYSICAL PLAN HOÀN CHỈNH:")
print("="*70)
(spark.read.parquet(partitioned_path)
  .filter((F.col("year") == 2024) & (F.col("month") == 6) & (F.col("city") == "HCM"))
  .select("amount")
  .agg(F.sum("amount"))
  .explain(True))

Trong plan, chú ý 3 dòng:
- `PartitionFilters: [...year=2024..., ...month=6...]` → Partition pruning
- `PushedFilters: [...EqualTo(city,HCM)]` → Predicate pushdown
- `ReadSchema: struct<city:string,amount:double>` → Column pruning (chỉ đọc cột cần để filter + tính sum)

---


---
## 8. Tổng kết

| Kỹ thuật | Cần làm gì | Khi nào hiệu quả |
|----------|-----------|------------------|
| **Format Parquet/ORC** | Ghi data bằng `.parquet()` thay vì `.csv()` | Luôn luôn, trừ khi cần human-readable |
| **Predicate Pushdown** | Dùng `.filter()` với điều kiện đơn giản, tránh UDF | Filter trên cột có index/statistics |
| **Column Pruning** | Dùng `.select("col_a", "col_b")` thay vì `select("*")` | Bảng có nhiều cột nhưng query chỉ cần vài cột |
| **Partition Pruning** | `partitionBy()` cột có cardinality thấp (year, region) | Query thường lọc theo cột partition |


### Quy trình tối ưu khi xây pipeline

1. **Format**: Parquet (mặc định) hoặc Delta/Iceberg nếu cần ACID.
2. **Partition**: chọn 1-2 cột cardinality thấp (year/month/region), kiểm tra mỗi partition ≥ 100MB.
3. **Compaction**: gộp file nhỏ định kỳ (tránh small file problem).
4. **Query**: luôn `select()` cụ thể, `filter()` sớm nhất có thể.
5. **Verify**: dùng `.explain(True)` để xác nhận `PartitionFilters`, `PushedFilters`, `ReadSchema` đều đúng.

### Bài tập

1. Tăng `N_ROWS` lên 10 triệu và đo lại — sự khác biệt sẽ rõ ràng hơn nhiều.
2. Thử partition theo `(year, month, city)` — quan sát số thư mục tạo ra.
3. Viết 1 query dùng UDF trong filter — xem trong `explain` filter có còn được pushdown không?
4. Ghi data với các `compression` khác nhau (`snappy`, `gzip`, `zstd`) và so sánh size + read time.

In [ ]:
# Cleanup
spark.sql("DROP DATABASE IF EXISTS demo_db CASCADE")
spark.stop()
print("Done!")